### 1. Import Libraries

In [1]:
import folium
import numpy as np
import pandas as pd

### 2. Data Preparation & Demand Simulation
Defining the logistic dataset for major cities in Java Island along with their geographical coordinates (`Latitude` & `Longitude`), and simulating annual demand values using a normal distribution ensured to be positive.

In [2]:
data = {
    'City': ['Jakarta', 'Bandung', 'Semarang', 'Surabaya', 'Yogyakarta', 'Malang', 'Serang'],
    'Latitude': [-6.2088, -6.9175, -6.9666, -7.2575, -7.7956, -7.9666, -6.1154],
    'Longitude': [106.8456, 107.6191, 110.4197, 112.7521, 110.3695, 112.6326, 106.1503]
}

df = pd.DataFrame(data)

np.random.seed(77)
df['Demand_Ton'] = np.abs(np.random.normal(1000, 6000, size=len(df))).astype(int)

display(df)

,City,Latitude,Longitude,Demand_Ton
0,Jakarta,-6.2088,106.8456,2346
1,Bandung,-6.9175,107.6191,4969
2,Semarang,-6.9666,110.4197,2478
3,Surabaya,-7.2575,112.7521,3446
4,Yogyakarta,-7.7956,110.3695,3798
5,Malang,-7.9666,112.6326,10848
6,Serang,-6.1154,106.1503,2653


### 3. Center of Gravity (CoG) Mathematical Calculation
Calculating the optimal facility (warehouse) coordinate point using the Center of Gravity mathematical formula:
$$X_{opt} = \frac{\sum (Lon_i \times Demand_i)}{\sum Demand_i}, \quad Y_{opt} = \frac{\sum (Lat_i \times Demand_i)}{\sum Demand_i}$$

In [3]:
total_demand = df['Demand_Ton'].sum()

optimal_latitude = (df['Latitude'] * df['Demand_Ton']).sum() / total_demand
optimal_longitude = (df['Longitude'] * df['Demand_Ton']).sum() / total_demand

### 4. Interactive Map Visualization
Mapping origin facility locations (cities) with circle markers proportional to their demand scale, and highlighting the optimal warehouse point (CoG) using a red star icon via the Folium library.

In [4]:
m = folium.Map(
    location=[-7.2575, 110.4197], 
    zoom_start=7, 
    tiles=(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}'
    ),
    attr=(
        'Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap,'
        ' iPC, NRCAN, Esri Japan, METI, Esri China (Hong Kong), Esri (Thailand),'
        ' TomTom, 2012'
    ),
)

for idx, row in df.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=row['Demand_Ton'] / 400,
        popup=f"<b>{row['City']}</b><br>Demand: {row['Demand_Ton']} Ton",
        color='blue',
        fill=True,
        fill_color='blue'
    ).add_to(m)

folium.Marker(
    location=[optimal_latitude, optimal_longitude],
    popup=(
        '<b>Center of Gravity (CoG)</b><br>Latitude:'
        f' {optimal_latitude:.4f}<br>Longitude: {optimal_longitude:.4f}'
    ),
    icon=folium.Icon(color='red', icon='star', prefix='fa')
).add_to(m)

m.save('center_of_gravity_map.html')